
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

#### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENROUTER_API_KEY"]=os.getenv("OPENROUTER_API_KEY", "")

In [2]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(model = "stepfun/step-3.5-flash:free")
model

d:\Agentic AI 101\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


ChatOpenRouter(profile={'max_input_tokens': 256000, 'max_output_tokens': 256000, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<openrouter.sdk.OpenRouter object at 0x000001A88244CD70>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com/oss', app_title='langchain', model_name='stepfun/step-3.5-flash:free', model_kwargs={})

In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    director: str = Field(description="The director of the movie")
    release_year: int = Field(description="The year the movie was released")
    rating: float = Field(description="The rating of the movie out of 10")

In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatOpenRouter(profile={'max_input_tokens': 256000, 'max_output_tokens': 256000, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<openrouter.sdk.OpenRouter object at 0x000001A88244CD70>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com/oss', app_title='langchain', model_name='stepfun/step-3.5-flash:free', model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'release_year': {'description': 'The year the movie was released', 'type': 'integer'}, 'rating': {'description': 'The rating of the movie out of 10', 'type': 'number'

In [5]:
model.invoke("Provide details about the movie Shawshank Redemption.")

AIMessage(content='Of course. Here is a detailed overview of the critically acclaimed film **"The Shawshank Redemption"**.\n\n### **Basic Information**\n*   **Title:** The Shawshank Redemption\n*   **Year:** 1994\n*   **Director:** Frank Darabont\n*   **Screenplay:** Frank Darabont (based on the novella *Rita Hayworth and Shawshank Redemption* by Stephen King)\n*   **Genre:** Drama, Prison Film\n*   **Runtime:** 142 minutes\n*   **Setting:** Shawshank State Penitentiary, a fictional Maine prison, from 1947 to the mid-1960s.\n\n---\n\n### **Plot Summary (No Major Spoilers)**\nThe story is narrated by **Ellis "Red" Redding** (Morgan Freeman), a longtime inmate and the prison\'s "go-to guy" for contraband. The film follows the arrival of a new prisoner, **Andy Dufresne** (Tim Robbins), a quiet, intelligent banker convicted of the double murder of his wife and her lover. Andy maintains his innocence.\n\nOver nearly two decades, Andy endures the brutal realities of prison life under the cor

In [8]:
res = model_with_structure.invoke("Provide details about the movie Shawshank Redemption.")
print(res)

title='The Shawshank Redemption' director='Frank Darabont' release_year=1994 rating=9.3


##### Message output alongside parsed structure

In [9]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """"A movie with details"""
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    release_year: int = Field(..., description="The year the movie was released")
    rating: float = Field(..., description="The rating of the movie out of 10")
    
model_with_structure = model.with_structured_output(Movie, include_raw = True)

res = model_with_structure.invoke("Provide details about the movie Shawshank Redemption.")
print(res)

{'raw': AIMessage(content='I\'ll provide details about "The Shawshank Redemption" movie.', additional_kwargs={'reasoning_content': 'The user is asking for details about "Shawshank Redemption". This is a well-known movie. I need to provide details using the Movie function. Let me think about the required parameters:\n\n- title: "The Shawshank Redemption"\n- director: Frank Darabont\n- release_year: 1994\n- rating: 9.3 (IMDb rating, commonly cited)\n\nThese are all required parameters. I should make sure to use the correct title format. The official title is "The Shawshank Redemption". I\'ll provide this information.\n\nLet me call the Movie function with these details.', 'reasoning_details': [{'type': 'reasoning.text', 'text': 'The user is asking for details about "Shawshank Redemption". This is a well-known movie. I need to provide details using the Movie function. Let me think about the required parameters:\n\n- title: "The Shawshank Redemption"\n- director: Frank Darabont\n- release_

##### Nested structures

In [10]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name: str = Field(..., description="The name of the actor")
    role: str = Field(..., description="The role of the actor in the movie")
    
class MovieDetails(BaseModel):
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    release_year: int = Field(..., description="The year the movie was released")
    genre: str = Field(..., description="The genre of the movie")
    budget: int = Field(..., description="The budget of the movie in USD")
    cast: list[Actor] = Field(..., description="A list of actors in the movie")
    
model_with_structure = model.with_structured_output(MovieDetails)

res = model_with_structure.invoke("Provide detailed information about the movie Inception.")
print(res)

title='Inception' director='Christopher Nolan' release_year=2010 genre='Science fiction' budget=160000000 cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Miles'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Tom Berenger', role='Browning'), Actor(name='Pete Postlethwaite', role='Maurice Fischer'), Actor(name='Dileep Rao', role='Yusuf')]


### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [11]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'title': 'The Avengers', 'year': 2012, 'director': 'Joss Whedon', 'rating': 8}

In [17]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'title': 'The Avengers',
 'year': 2012,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Stellan Skarsgård', 'role': 'Erik Selvig'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'budget': 220000000}

In [14]:
model.profile

{'max_input_tokens': 256000,
 'max_output_tokens': 256000,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [20]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='6c16eab3-c373-40c1-87e5-39b7143a7f68'),
  AIMessage(content="I'll extract the contact information from the provided text.", additional_kwargs={'reasoning_content': 'The user is asking me to extract contact info from a string that contains "John Doe, john@example.com, (555) 123-4567". I have a function called ContactInfo that can create a contact information object with name, email, and phone. The parameters are name (string), email (string), and phone (string), all required.\n\nI need to parse the input string and extract the three pieces of information:\n- Name: "John Doe"\n- Email: "john@example.com"\n- Phone: "(555) 123-4567"\n\nThese seem clearly separated by commas. I can simply extract them directly.\n\nNow I need to call the ContactInfo function with these values.', 'reasoning_details': [{'type': 'reasoning.text', 'text': 'Th

In [21]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [22]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [23]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')